In [1]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv", parse_dates = ['signup_date'], dtype = {'birth_year': 'Int64'})
products = pd.read_csv("../data/raw/products.csv", parse_dates = ['launch_date', 'discontinued_date'])
orders = pd.read_csv("../data/raw/orders.csv", parse_dates = ['order_ts', 'promised_date', 'delivered_date'], dtype = {'session_id' : 'Int64'})
order_items = pd.read_csv("../data/raw/order_items.csv", parse_dates = ['return_date'])
user_events = pd.read_csv("../data/raw/user_events.csv", parse_dates = ['event_ts'], dtype = {'customer_id': 'Int64', 'product_id': 'Int64'})
marketing_spend = pd.read_csv("../data/raw/marketing_spend.csv", parse_dates = ['spend_date'], dtype = {'platform_conversions' : 'Int64'})

tables = {"customers": customers,
          "products": products,
          "orders": orders,
          "order_items": order_items,
          "user_events": user_events,
          "marketing_spend": marketing_spend
         }

In [2]:
customers.head(10)

,customer_id,email_hash,signup_date,country,city,gender,birth_year,acquisition_source,marketing_opt_in
0,1,b57bf7094ed8,2023-12-28,LT,Test,NaN,<NA>,direct,0
1,2,8b7ec82f60c6,2024-01-01,LT,Šiauliai,Male,1999,organic_search,1
2,3,bbfae928f872,2024-01-02,LT,Klaipėda,Male,<NA>,direct,1
3,4,7548ae3fa95a,2024-01-02,LT,Kaunas,Female,1996,paid_social,1
4,5,528728837f93,2024-01-02,LT,Siauliai,Female,1991,paid_social,1
5,6,2ba1a69ceef4,2024-01-03,LT,Panevėžys,Female,1989,direct,0
6,7,7081b25e2e7e,2024-01-03,LT,Vilnius,Female,1979,email,0
7,8,d210317e87a9,2024-01-04,LT,Marijampolė,Male,1987,organic_search,1
8,9,c3c4cba41af4,2024-01-04,LT,Vilnius,Female,1993,direct,0
9,10,ffacef68e448,2024-01-05,EE,Narva,Male,1990,paid_search,1


## My questions
1. Are all 'signup_date' values within the data period stated in the data dictionary?
2. Which values in 'gender' and 'city' mean the same thing but are written differently?
4. Can one person have more than one account (for example, the same email_hash)?

In [3]:
products.head(10)

,product_id,product_name,category,subcategory,brand,list_price,unit_cost,launch_date,discontinued_date
0,1,Lumea Reed Diffuser 100ml,Home & Kitchen,Decor,Lumea,19.99,84.00,2023-09-03,NaT
1,2,Strata Lightweight Running Jacket,Sports & Outdoors,Running,Strata,69.99,41.13,2023-09-03,NaT
2,3,Fjord Wear Men's Down Parka,Fashion,Outerwear,Fjord Wear,249.99,101.69,2023-09-04,NaT
3,4,Naturo Relaxing Massage Oil 200ml,Beauty & Care,Wellness,Naturo,14.99,8.56,2023-09-04,NaT
4,5,Pixora Indoor Security Camera 2K,Electronics,Smart Home,Pixora,49.99,35.03,2023-09-04,NaT
5,6,Silka Ionic Hair Dryer 2200W,Beauty & Care,Haircare,Silka,59.99,25.24,2023-09-04,NaT
6,7,Panela Stainless Steel Pot Set 5-Piece,Home & Kitchen,Cookware,Panela,139.99,72.36,2023-09-06,NaT
7,8,Whiskr Wet Cat Food 12x85g,Pets,Food,Whiskr,13.99,10.58,2023-09-06,NaT
8,9,Aktiva Football Size 5,Sports & Outdoors,Team Sports,Aktiva,24.99,15.40,2023-09-07,NaT
9,10,Dermia Men's Face Moisturizer 75ml,Beauty & Care,Skincare,Dermia,19.99,8.96,2023-09-07,NaT


## My questions
1. Are there products with 'unit_cost' higher than 'list_price'? How many? Why?
2. Were any products sold before their 'launch_date' or after their 'discontinued_date'?
3. Are 'category', 'subcategory' and 'brand' values written consistently (no different spellings of the same value)?
4. Does every 'subcategory' belong to only one 'category'?

In [4]:
orders.insert(4, "order_date_local", orders["order_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
orders.head(10)

,order_id,customer_id,session_id,order_ts,order_date_local,status,coupon_code,payment_method,delivery_method,shipping_fee,order_total,promised_date,delivered_date
0,1,2,15,2024-01-01 14:18:20,2024-01-01,completed,NaN,bank_link,parcel_locker,2.99,27.98,2024-01-03,2024-01-03
1,2,1,32,2024-01-02 07:12:30,2024-01-02,cancelled,NaN,card,pickup,0.00,0.01,2024-01-02,NaT
2,3,1,34,2024-01-02 07:40:02,2024-01-02,cancelled,NaN,card,pickup,0.00,0.01,2024-01-02,NaT
3,4,3,39,2024-01-02 10:45:31,2024-01-02,completed,NaN,card,courier,0.00,144.98,2024-01-03,2024-01-03
4,5,4,45,2024-01-02 14:52:26,2024-01-02,completed,WELCOME10,card,parcel_locker,2.99,24.57,2024-01-04,2024-01-04
5,6,5,48,2024-01-02 17:06:21,2024-01-02,completed,WELCOME10,bank_link,parcel_locker,0.00,181.76,2024-01-04,2024-01-04
6,7,6,65,2024-01-03 00:48:14,2024-01-03,cancelled,WELCOME10,bank_link,courier,4.99,27.48,2024-01-04,NaT
7,8,1,81,2024-01-03 12:05:47,2024-01-03,completed,NaN,card,pickup,0.00,0.01,2024-01-04,2024-01-03
8,9,7,83,2024-01-03 12:12:26,2024-01-03,completed,NaN,card,courier,0.00,52.98,2024-01-05,2024-01-05
9,10,9,139,2024-01-04 15:35:51,2024-01-04,completed,NaN,card,parcel_locker,2.99,38.96,2024-01-06,2024-01-06


## My questions
1. Are all 'order_ts', 'promised_date', 'delivered_date' values within the data period stated in the data dictionary? ('order_ts' is UTC time)
2. Are there any cancelled orders that have a 'delivered_date'?
3. Are there any orders whose local order date is later than 'promised_date' or 'delivered_date'?
4. Are there any orders that don't look like real customer orders ('order_total' close to 0)? Who placed them?

In [5]:
order_items.head(10)

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,is_returned,return_reason,return_date
0,1,1,31,1,24.99,0,0,NaN,NaT
1,2,2,111,1,0.01,0,0,NaN,NaT
2,3,3,136,1,0.01,0,0,NaN,NaT
3,4,4,7,1,119.99,0,0,NaN,NaT
4,5,4,85,1,24.99,0,0,NaN,NaT
5,6,5,78,1,11.99,10,0,NaN,NaT
6,7,5,132,1,11.99,10,0,NaN,NaT
7,8,6,7,1,119.99,10,0,NaN,NaT
8,9,6,41,1,19.99,10,1,changed_mind,2024-01-06
9,10,6,109,1,44.99,10,0,NaN,NaT


## My questions
1. Can the same 'product_id' appear more than once within one 'order_id'?
2. Are 'quantity', 'unit_price' and 'discount_pct' within realistic ranges?
3. When 'is_returned' = 1, are 'return_reason' and 'return_date' always filled in? When 'is_returned' = 0, are they always empty?

In [6]:
user_events.insert(6, "event_date_local", user_events["event_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
user_events.head(10)

,event_id,session_id,visitor_id,customer_id,event_type,event_ts,event_date_local,device,traffic_source,product_id
0,1,1,7940527645,<NA>,view_item,2024-01-01 05:15:03,2024-01-01,mobile,organic_search,109
1,2,1,7940527645,<NA>,view_item,2024-01-01 05:18:46,2024-01-01,mobile,organic_search,111
2,3,1,7940527645,<NA>,view_item,2024-01-01 05:18:52,2024-01-01,mobile,organic_search,41
3,4,1,7940527645,<NA>,view_item,2024-01-01 05:19:04,2024-01-01,mobile,organic_search,7
4,5,1,7940527645,<NA>,view_item,2024-01-01 05:20:27,2024-01-01,mobile,organic_search,138
5,6,1,7940527645,<NA>,view_item,2024-01-01 05:21:04,2024-01-01,mobile,organic_search,131
6,7,2,2772890305,<NA>,view_item,2024-01-01 05:57:53,2024-01-01,mobile,affiliate,36
7,8,3,7286947083,<NA>,view_item,2024-01-01 06:09:33,2024-01-01,mobile,organic_search,84
8,9,4,7282267126,<NA>,view_item,2024-01-01 07:32:25,2024-01-01,desktop,organic_search,138
9,10,5,2227241223,<NA>,view_item,2024-01-01 07:56:38,2024-01-01,desktop,organic_search,17


## My questions
1. Are all 'event_ts' (UTC) values within the data period stated in the data dictionary?
2. Can one 'session_id' have more than one 'visitor_id', 'device' or 'traffic_source'?
3. For which 'event_type' values is 'customer_id' filled in, and is it always filled in for purchase events?
4. Are there sessions or visitors with unusual activity (very many events, very little time between events) that could be bots?
5. Is 'product_id' filled in only for 'view_item' and 'add_to_cart' events?

In [7]:
marketing_spend.head(10)

,spend_date,channel,campaign_type,impressions,clicks,spend_eur,platform_conversions
0,2024-01-01,affiliate,commission,528,2,0.00,0
1,2024-01-01,email,newsletter_platform,7,6,1.58,0
2,2024-01-01,organic_search,seo_retainer,1076,29,14.52,<NA>
3,2024-01-01,paid_search,brand,7,2,0.27,0
4,2024-01-01,paid_search,non_brand,83,4,1.87,0
5,2024-01-01,paid_social,prospecting,223,2,0.57,0
6,2024-01-01,paid_social,retargeting,68,1,0.38,0
7,2024-01-02,affiliate,commission,606,2,0.00,0
8,2024-01-02,email,newsletter_platform,7,2,1.58,0
9,2024-01-02,organic_search,seo_retainer,911,30,14.52,<NA>


## My questions
1. Are all 'spend_date' values within the data period stated in the data dictionary?
2. Are 'channel' and 'campaign_type' values written consistently (no different spellings of the same value)?
3. Is 'platform_conversions' empty only for 'organic_search' rows?
4. Does every 'spend_date' have exactly one row for each 'channel' and 'campaign_type' combination (no missing and no duplicate rows)?
5. Are there any rows where 'clicks' are higher than 'impressions'?
6. Does affiliate 'spend_eur' appear only some time after the first affiliate clicks?
7. Is the daily 'spend_eur' for email and organic_search the same on every day within a month?

## Check dtypes

In [8]:
for name, df in tables.items():
    print(f"=== Table: {name} ===")
    print(f"{df.dtypes}")
    print(f"\n")
    

=== Table: customers ===
customer_id                    int64
email_hash                       str
signup_date           datetime64[us]
country                          str
city                             str
gender                           str
birth_year                     Int64
acquisition_source               str
marketing_opt_in               int64
dtype: object


=== Table: products ===
product_id                    int64
product_name                    str
category                        str
subcategory                     str
brand                           str
list_price                  float64
unit_cost                   float64
launch_date          datetime64[us]
discontinued_date    datetime64[us]
dtype: object


=== Table: orders ===
order_id                     int64
customer_id                  int64
session_id                   Int64
order_ts            datetime64[us]
order_date_local    datetime64[us]
status                         str
coupon_code                   

## Profiling function

In [136]:
def profile(df, name, key):
    print(f"===== {name} =====")
    print(f"\n***** Shape *****\n")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")
    print(f"***** Dtypes *****\n")
    print(df.dtypes)
    print(f"\n***** Unique Key *****\n")
    print(f"How many duplicates in {', '.join(key)}? {df.duplicated(subset=key).sum()}")
    print(f"How many missing values in {', '.join(key)}? {df[key].isna().any(axis=1).sum()}")
    print(f"How many fully duplicated rows? {df.duplicated().sum()}")
    print(f"\n***** NULL values in columns *****\n")
    display(pd.DataFrame({'Missing values': df.isna().sum(), '% Missing': (df.isna().mean() * 100).round(2)}))
    print(f"\n***** MIN, MAX, PERCENTILES *****\n")
    display(df.select_dtypes(include="number").describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)
    print(f"\n***** CATEGORICAL VARIABLES *****\n")
    
    text_cols = df.select_dtypes(exclude=["number", "datetime"]).columns
    for col in text_cols:
        n_unique = df[col].nunique()
        n_clean = df[col].str.strip().str.lower().nunique()
        print(f"\n----- {col}: {n_unique} unique -> {n_clean} after strip/lower -----\n")
        if n_unique <= 20:
            display(df[col].value_counts(dropna=False).to_frame("count"))
    print(f"\n***** DATE RANGE *****\n")
    display(df.select_dtypes(include="datetime").agg(["min", "max"]).T)
    

profile(customers, "Customers", ["customer_id"])

===== Customers =====

***** Shape *****

Rows: 7483, Columns: 9

***** Dtypes *****

customer_id                    int64
email_hash                       str
signup_date           datetime64[us]
country                          str
city                             str
gender                           str
birth_year                     Int64
acquisition_source               str
marketing_opt_in               int64
dtype: object

***** Unique Key *****

How many duplicates in customer_id? 0
How many missing values in customer_id? 0
How many fully duplicated rows? 0

***** NULL values in columns *****



,Missing values,% Missing
customer_id,0,0.00
email_hash,0,0.00
signup_date,0,0.00
country,0,0.00
city,439,5.87
gender,483,6.45
birth_year,1009,13.48
acquisition_source,0,0.00
marketing_opt_in,0,0.00



***** MIN, MAX, PERCENTILES *****



,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
customer_id,7483.0,3742.0,2160.300365,1.0,75.82,375.1,1871.5,3742.0,5612.5,7108.9,7408.18,7483.0
birth_year,6474.0,1984.959376,11.947787,1900.0,1955.73,1964.0,1977.0,1986.0,1994.0,2002.0,2006.0,2026.0
marketing_opt_in,7483.0,0.57664,0.494124,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0



***** CATEGORICAL VARIABLES *****


----- email_hash: 7427 unique -> 7427 after strip/lower -----


----- country: 3 unique -> 3 after strip/lower -----



...

,count
gender,
F,3107
M,2527
Female,685
Male,593
NaN,483
X,69
Other,19



----- acquisition_source: 7 unique -> 7 after strip/lower -----



,count
acquisition_source,
organic_search,2104
paid_social,1433
paid_search,1281
direct,1052
referral,646
affiliate,563
email,404



***** DATE RANGE *****



,min,max
signup_date,2023-12-28,2026-09-30


## Next steps
- Run profile() for products (then other tables)
- Write answers under "My questions"
- Add findings to docs/data_issues.md